In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/hadiyanto/IdeaProjects/pse/Entwicklungsdaten.tsv", sep="\t")
df1 = pd.read_csv("/Users/hadiyanto/IdeaProjects/pse/negative_tweets.tsv", sep="\t", names=["c_text","hate","type","score"])

In [3]:
joined_frames = [df, df1]
result = pd.concat(joined_frames)
#result.to_csv('/Users/hadiyanto/Desktop/result.tsv', sep="\t")

In [4]:
text = result.iloc[0:, 1]

In [5]:
print (text)

0       @Laika66753508 @MartinaWeiss48 @reitschuster M...
1       @manaf12hassan Die Karte muss in Gesicht von F...
2       @Tino_Chrupalla @Karl_Lauterbach Wenn Sie kack...
3                            @DemokratDer @manuelak62 666
4       @_agronym Junge was für Verfolgung, die befeue...
                              ...                        
1911    Kein Staat der Welt hat das Recht, Menschen al...
1912    @robinalexander_ Gender als Ersatzreligion. Ho...
1913    Sagte Jesus Christus nicht auch, |LBR| Du soll...
1914    @KokoLores20 @krippmarie Ein richtiges Zeichen...
1915    @Hartes_Geld ,Honecker‘Merkel macht uns zur ,D...
Name: c_text, Length: 10048, dtype: object


In [6]:
tempArray = []
for row in range(len(df1)):
    #print(df1.iloc[row, 2])
    selected_row =df1.iloc[row, 2]
    if selected_row == "INSULT" or selected_row == "ABUSE":
        tempArray.append(1)
    else:
        tempArray.append(0)
#print(tempArray)

In [7]:
df1['label'] = pd.Series(tempArray)

In [8]:
df1

,c_text,hate,type,score,label
0,@Martin28a Sie haben ja auch Recht. Unser Twee...,OTHER,OTHER,-0.70,0
1,@spdde kein verläßlicher Verhandlungspartner. ...,OFFENSE,INSULT,-1.70,1
2,@milenahanm 33 bis 45 habe ich noch gar nicht ...,OFFENSE,PROFANITY,-1.00,0
3,@tagesschau Euere AfD Hetze wirkt. Da könnt ih...,OFFENSE,ABUSE,-1.00,1
4,"Deutsche Medien, Halbwahrheiten und einseitige...",OFFENSE,ABUSE,-1.60,1
...,...,...,...,...,...
1911,"Kein Staat der Welt hat das Recht, Menschen al...",OTHER,OTHER,-3.70,0
1912,@robinalexander_ Gender als Ersatzreligion. Ho...,OTHER,OTHER,-1.40,0
1913,"Sagte Jesus Christus nicht auch, |LBR| Du soll...",OTHER,OTHER,-1.30,0
1914,@KokoLores20 @krippmarie Ein richtiges Zeichen...,OFFENSE,ABUSE,-1.25,1


In [9]:
hs = (df.iloc[0: , 8 ].to_list() + df1.iloc[0: , 4].to_list())
#pint(hs)

In [10]:
import spacy
nlp = spacy.load("de_core_news_sm")

In [11]:
array =[]
for t in text[:]:
    doc = nlp(t)
    #print([(token.text, token.pos_) for token in doc if token.is_alpha and not token.is_stop])
    
    #normalize dataset
    #temp = [token.lemma_.lower() for token in doc if token.is_alpha]
    #print(temp)
    
    #without stop words
    temp = [token.lemma_.lower() for token in doc if token.is_alpha and not token.is_stop]
    #print(temp)

    array.append((" ".join(temp)))

#print (array)

In [12]:
df = pd.DataFrame ({'Text': array , 'Hatespeech': hs })
print (df)

                                                    Text  Hatespeech
0                               liebe bilden anscheinend           0
1            karte gesicht frau merkel herr spahn ziehen           0
2      kackbraun scheiße blau übermalen anstreicher c...           1
3                                                                  0
4      jung verfolgung befeuer eigentumswohnung querd...           0
...                                                  ...         ...
10043  staat welt mensch strafe töten todesstrafe wid...           0
10044  gend ersatzreligion hohenpriesterinn ziehen vo...           0
10045  jesus christus sollen gott mal schauen gott wo...           0
10046  richtig zeichen nachbar schleichend islamisier...           1
10047  honecker merkel ddr klage annehmen zeitung gle...           1

[10048 rows x 2 columns]


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.Text, 
    df.Hatespeech, 
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022, #-> same order of dataset
    stratify=df.Hatespeech
)

In [10]:
print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

Shape of X_train:  (6505,)
Shape of X_test:  (1627,)


In [11]:
y_train.value_counts()

0    5567
1     938
Name: Hatespeech, dtype: int64

In [12]:
y_test.value_counts()

0    1392
1     235
Name: Hatespeech, dtype: int64

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
import xgboost as xgb

#1. create a pipeline object
clf = Pipeline([
     ('vectorizer_tfidf',TfidfVectorizer()),    
     #('KNN', KNeighborsClassifier()),
     #('SVM', SVC())   
     #('rfc', RandomForestClassifier()) 
     ('xgb', xgb.XGBClassifier(objective="binary:logistic", random_state=42)) 
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

# find accuracy scores
accuracy = accuracy_score(y_test, y_pred)
print("The accuracy of prediction is: ", accuracy)

              precision    recall  f1-score   support

           0       0.87      0.98      0.92      1392
           1       0.57      0.16      0.25       235

    accuracy                           0.86      1627
   macro avg       0.72      0.57      0.59      1627
weighted avg       0.83      0.86      0.83      1627

The accuracy of prediction is:  0.8610940381069453


In [14]:
#X_test[:100]
c =0
for i in X_test[:200]:
    print(c,"->", i)
    c+=1

0 -> oh risiko trägst angst alleine geimpft
1 -> klar dreckig verbrecher giftspritze bekommen höchstens kochsalzlösung diktator
2 -> russe deutsch pole überfallen aufteilen irgendwie
3 -> derzeit rimworld restaggro abbauen mission k inquisitor n schwer bolter kopfhörer durchrattern
4 -> eher weltbild passen einfach elementar unterschied ignorieren
5 -> praxis stiko empfehlung halten rate seriös kinderarzt experimentell impfstoff regulär zulassung kind verimpfen
6 -> logisch impfneid aufkommen solange möglichkeit impfen schlange versuchen schwanz packen kopf
7 -> religion wissenschaft beruhen paradoxon dennoch wünschenswert wünschenswerter euer braun schmutz allemal fckafd teamklimaschutz klimaschutz baerbock
8 -> lachen lachen laut eigentlich deutsche überhören lassen tagsüber erzählen glauben erfolgen theater kanzlerschaft bock reichen völlig bock
9 -> finden hashtag antifa ziemlich cool nehmen
10 -> baerbockvshabeck partei dumm unerträglich schrill nervend stimme wohltemperierter sym

In [15]:
#y_test[0:50]
c =0
for i in y_test[0:]:
    print(c,"->", i)
    c+=1

h=0
nh=0
for i in y_test:
    if i == 1:
        #print(i)
        h+=1
    if i == 0:
        #print(i)
        nh+=1
print("count of HS: ", h) 
print("count of NS: ", nh)
        

0 -> 0
1 -> 0
2 -> 0
3 -> 0
4 -> 0
5 -> 0
6 -> 0
7 -> 0
8 -> 0
9 -> 0
10 -> 0
11 -> 0
12 -> 0
13 -> 0
14 -> 0
15 -> 0
16 -> 1
17 -> 0
18 -> 0
19 -> 0
20 -> 0
21 -> 0
22 -> 1
23 -> 0
24 -> 0
25 -> 0
26 -> 0
27 -> 1
28 -> 0
29 -> 0
30 -> 0
31 -> 0
32 -> 0
33 -> 0
34 -> 0
35 -> 1
36 -> 0
37 -> 0
38 -> 0
39 -> 0
40 -> 0
41 -> 0
42 -> 1
43 -> 0
44 -> 0
45 -> 0
46 -> 0
47 -> 0
48 -> 0
49 -> 0
50 -> 0
51 -> 1
52 -> 0
53 -> 0
54 -> 1
55 -> 0
56 -> 0
57 -> 0
58 -> 0
59 -> 0
60 -> 0
61 -> 0
62 -> 0
63 -> 0
64 -> 0
65 -> 0
66 -> 0
67 -> 1
68 -> 0
69 -> 0
70 -> 0
71 -> 0
72 -> 0
73 -> 0
74 -> 0
75 -> 0
76 -> 0
77 -> 0
78 -> 0
79 -> 0
80 -> 0
81 -> 0
82 -> 0
83 -> 1
84 -> 0
85 -> 0
86 -> 0
87 -> 0
88 -> 0
89 -> 0
90 -> 1
91 -> 0
92 -> 0
93 -> 0
94 -> 0
95 -> 0
96 -> 1
97 -> 0
98 -> 1
99 -> 1
100 -> 0
101 -> 0
102 -> 0
103 -> 1
104 -> 0
105 -> 0
106 -> 0
107 -> 0
108 -> 0
109 -> 0
110 -> 0
111 -> 0
112 -> 0
113 -> 0
114 -> 0
115 -> 0
116 -> 0
117 -> 0
118 -> 0
119 -> 0
120 -> 1
121 -> 0
122 -> 0
123

In [ ]:
#y_pred[0:50]
c =0
for i in y_pred[0:]:
    print(c,"->", i)
    c+=1

h=0
nh=0
for i in y_pred:
    if i == 1:
        #print(i)
        h+=1
    if i == 0:
        #print(i)
        nh+=1
print("count of HS: ", h) 
print("count of NS: ", nh)    